In [12]:
import numpy as np
import tensorflow as tf

from sklearn.datasets import make_moons
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score

In [13]:
np.random.seed(42)
tf.random.set_seed(42)

X, y = make_moons(
    n_samples=1200,
    noise=0.25,
    random_state=42
)

X_train_val, X_test, y_train_val, y_test = train_test_split(X, y, test_size=0.20, random_state=42, stratify=y)
X_train, X_val, y_train, y_val = train_test_split(X_train_val, y_train_val, test_size=0.20, random_state=42, stratify=y_train_val)

In [14]:
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_val = scaler.transform(X_val)
X_test = scaler.transform(X_test)

In [15]:
def build_model():
    model = tf.keras.Sequential([
        tf.keras.layers.Input(shape=(2,)),
        tf.keras.layers.Dense(32, activation="relu"),
        tf.keras.layers.Dense(16, activation="relu"),
        tf.keras.layers.Dense(1, activation="sigmoid")
    ])

    return model

In [16]:
batch_size = 32
epochs = 80

steps_per_epoch = int(np.ceil(len(X_train) / batch_size))

fixed_optimizer = tf.keras.optimizers.Adam(
    learning_rate=0.001
)

lr_schedule = tf.keras.optimizers.schedules.ExponentialDecay(
    initial_learning_rate=0.003,
    decay_steps=steps_per_epoch * 10,
    decay_rate=0.9,
    staircase=True
)

decay_optimizer = tf.keras.optimizers.Adam(learning_rate=lr_schedule)

plateau_optimizer = tf.keras.optimizers.Adam(learning_rate=0.003)

plateau_callback = tf.keras.callbacks.ReduceLROnPlateau(
    monitor="val_loss",
    factor=0.5,
    patience=5,
    min_delta=0.00001,
    min_lr=0.00001,
    verbose=1
)

strategies = [
    ("Fixed Adam", fixed_optimizer, []),
    ("Exponential Decay", decay_optimizer, []),
    ("ReduceLROnPlateau", plateau_optimizer, [plateau_callback])
]

In [17]:
results = {}

for name, optimizer, callbacks in strategies:
    print(f"\nTraining : {name}")

    tf.keras.utils.set_random_seed(42)

    model=build_model()

    model.compile(
        optimizer=optimizer,
        loss="binary_crossentropy",
        metrics=['accuracy']
    )

    history = model.fit(
        X_train,
        y_train,
        validation_data=(X_val, y_val),
        epochs=epochs,
        batch_size=batch_size,
        callbacks=callbacks,
        verbose=0
    )

    test_loss, test_accuracy = model.evaluate(
        X_test, y_test, verbose=0
    )

    results[name] = {
        "final_train_loss" : history.history['loss'][-1],
        "final_val_loss" : history.history['val_loss'][-1],
        "test_loss" : test_loss,
        "test_accuracy" : test_accuracy
    }

print("\nFINAL COMPARISION")
for name, metrics in results.items():
    print(f"\n{name}")

    for metric_name, value in metrics.items():
        print(f"{metric_name} : {value:.4f}")


Training : Fixed Adam

Training : Exponential Decay

Training : ReduceLROnPlateau

Epoch 40: ReduceLROnPlateau reducing learning rate to 0.001500000013038516.

Epoch 46: ReduceLROnPlateau reducing learning rate to 0.000750000006519258.

FINAL COMPARISION

Fixed Adam
final_train_loss : 0.1753
final_val_loss : 0.1335
test_loss : 0.1443
test_accuracy : 0.9458

Exponential Decay
final_train_loss : 0.1667
final_val_loss : 0.1296
test_loss : 0.1374
test_accuracy : 0.9500

ReduceLROnPlateau
final_train_loss : 0.1656
final_val_loss : 0.1287
test_loss : 0.1374
test_accuracy : 0.9458
